# 调试框选位置问题

In [ ]:
import torch
import os
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torchvision.transforms as T
import sys
sys.path.append(r'D:\\AI\\Git\\DEIMv2')
from engine.core import YAMLConfig

device = torch.device('cpu')

# 路径
CONFIG = r'D:\\AI\\Git\\DEIMv2\\configs\\deimv2\\deimv2_hgnetv2_atto_helmet_cpu3.yml'
MODEL = r'D:\\AI\\Git\\DEIMv2\\outputs\\deimv2_hgnetv2_atto_helmet_gpu_3060\\best_stg1.pth'
TEST_IMAGE = r'D:\\AI\\Datasets\\sy-person\\中秋1-H3（90022）防疲劳-20260320092859.png'

# 加载模型
cfg = YAMLConfig(CONFIG)
eval_size = cfg.yaml_cfg.get('eval_spatial_size', [640, 640])
model = cfg.model

checkpoint = torch.load(MODEL, map_location='cpu')
state_dict = checkpoint.get('model', checkpoint)
new_state_dict = {k[7:] if k.startswith('module.') else k: v for k, v in state_dict.items()}
model.load_state_dict(new_state_dict, strict=False)
model = model.to(device)
model.eval()

postprocessor = cfg.postprocessor
postprocessor.eval_spatial_size = eval_size
print('模型加载完成')

In [ ]:
# 加载图像
orig_image = Image.open(TEST_IMAGE).convert('RGB')
orig_w, orig_h = orig_image.size
print(f'原图尺寸: {orig_w}x{orig_h}')

# 预处理
image = orig_image.resize(tuple(eval_size))
tensor = T.ToTensor()(image)
tensor = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(tensor)
tensor = tensor.unsqueeze(0).to(device)

# 推理
with torch.no_grad():
    outputs = model(tensor)

# 后处理
orig_target_sizes = torch.tensor([[orig_h, orig_w]]).to(device)
results = postprocessor(outputs, orig_target_sizes)[0]

# 打印前5个检测结果
boxes = results['boxes'].cpu().numpy()
scores = results['scores'].cpu().numpy()
labels = results['labels'].cpu().numpy()

print(f'\n前5个检测结果（阈值>0.05）:\n')
count = 0
for box, score, label in zip(boxes, scores, labels):
    if score >= 0.05 and count < 5:
        x1, y1, x2, y2 = box
        print(f'  类别{int(label)}: 框=[{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}], 置信度={score:.3f}')
        print(f'    -> 框宽度={x2-x1:.1f}, 高度={y2-y1:.1f}')
        count += 1

In [ ]:
# 可视化前3个检测结果，检查框位置
result_image = orig_image.copy()
draw = ImageDraw.Draw(result_image)

count = 0
for box, score, label in zip(boxes, scores, labels):
    if score >= 0.05 and count < 3:
        x1, y1, x2, y2 = [int(x) for x in box]
        color = '#00FF00' if int(label) == 0 else '#FF0000'
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1, y1-20), f'{int(label)}:{score:.2f}', fill=color)
        count += 1

plt.figure(figsize=(15, 10))
plt.imshow(result_image)
plt.title('前3个检测框位置检查')
plt.axis('off')
plt.show()

print('\n如果框位置不对，请描述:')
print('  1. 框太小（集中在图像中间）')
print('  2. 框位置偏移（整体向左/右/上/下偏移）')
print('  3. 框的比例不对（宽高比异常）')